In [6]:
# Load the transactions dataset
import pandas as pd
import numpy as np

# Path to the dataset
data_path = '../Data/dataset_transacciones/hey_transacciones.csv'

# Load with date parsing
df = pd.read_csv(data_path, parse_dates=['fecha_hora'])

# Display basic info
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(df.head())

# Set reference date for recency as max fecha_hora
ref_date = df['fecha_hora'].max()
print(f"Reference date for recency: {ref_date}")

# Handle nulls: fill numeric nulls with 0, categorical with empty string if needed
numeric_cols = ['monto', 'cashback_generado', 'meses_diferidos']
df[numeric_cols] = df[numeric_cols].fillna(0)
df['motivo_no_procesada'] = df['motivo_no_procesada'].fillna('')
df['comercio_nombre'] = df['comercio_nombre'].fillna('') # ¿Put null comerce name?k
df['dispositivo'] = df['dispositivo'].fillna('')

Dataset shape: (802384, 22)
Columns: ['transaccion_id', 'user_id', 'producto_id', 'fecha_hora', 'tipo_operacion', 'canal', 'monto', 'comercio_nombre', 'categoria_mcc', 'ciudad_transaccion', 'estatus', 'motivo_no_procesada', 'intento_numero', 'meses_diferidos', 'cashback_generado', 'descripcion_libre', 'hora_del_dia', 'dia_semana', 'es_internacional', 'dispositivo', 'patron_uso_atipico', 'es_dato_sintetico']
   transaccion_id    user_id   producto_id          fecha_hora  \
0  TXN-0000000055  USR-00001  PRD-00000002 2025-01-15 14:17:42   
1  TXN-0000000048  USR-00001  PRD-00000001 2025-01-17 00:31:56   
2  TXN-0000000018  USR-00001  PRD-00000001 2025-01-17 22:48:23   
3  TXN-0000000043  USR-00001  PRD-00000001 2025-01-19 11:10:43   
4  TXN-0000000009  USR-00001  PRD-00000001 2025-02-15 07:03:50   

     tipo_operacion        canal    monto comercio_nombre  \
0            compra      app_ios    33.88   DivertidoPark   
1  cargo_recurrente      app_ios   249.00       GamerPass   
2  cargo_

## Volume & Frequency Features

In [7]:
# Volume & Frequency Features
# Focus on purchases for spend-related features
purchases = df[df['tipo_operacion'] == 'compra'].copy()

# Add year-month for monthly aggregations
purchases['year_month'] = purchases['fecha_hora'].dt.to_period('M')

# Monthly spend sums per user
monthly_spend = purchases.groupby(['user_id', 'year_month'])['monto'].sum().reset_index()

# monthly_avg_spend: mean of monthly sums
monthly_avg_spend = monthly_spend.groupby('user_id')['monto'].mean().rename('monthly_avg_spend')

# txn_frequency: average monthly transaction count
monthly_txn_count = purchases.groupby(['user_id', 'year_month']).size().reset_index(name='count')
txn_frequency = monthly_txn_count.groupby('user_id')['count'].mean().rename('txn_frequency')

# avg_ticket_size: mean monto per purchase
avg_ticket_size = purchases.groupby('user_id')['monto'].mean().rename('avg_ticket_size')

# spend_volatility: coefficient of variation of monthly spends
# Only compute if at least 2 months of data
monthly_count = monthly_spend.groupby('user_id').size()
monthly_spend_stats = monthly_spend.groupby('user_id')['monto'].agg(['mean', 'std'])
monthly_spend_stats['spend_volatility'] = monthly_spend_stats['std'] / monthly_spend_stats['mean']
spend_volatility = monthly_spend_stats['spend_volatility'].where(monthly_count >= 2, 0).fillna(0)

# failed_txn_rate: proportion of failed transactions
failed_txn_rate = df.groupby('user_id')['estatus'].apply(lambda x: (x == 'no_procesada').mean()).rename('failed_txn_rate')

# retry_rate: proportion with intento_numero > 1
retry_rate = df.groupby('user_id')['intento_numero'].apply(lambda x: (x > 1).mean()).rename('retry_rate')

# dispute_rate: proportion with estatus == 'en_disputa'
dispute_rate = df.groupby('user_id')['estatus'].apply(lambda x: (x == 'en_disputa').mean()).rename('dispute_rate')

# Combine volume & frequency features
volume_freq_features = pd.concat([monthly_avg_spend, txn_frequency, avg_ticket_size, spend_volatility, failed_txn_rate, retry_rate, dispute_rate], axis=1).fillna(0)
print("Volume & Frequency features shape:", volume_freq_features.shape)
print(volume_freq_features.head())

Volume & Frequency features shape: (15025, 7)
           monthly_avg_spend  txn_frequency  avg_ticket_size  \
user_id                                                        
USR-00001        1215.000000       2.818182       431.129032   
USR-00002        1468.359091       3.909091       375.626744   
USR-00003        1638.176364       3.909091       419.068372   
USR-00004        5661.518182       2.454545      2306.544444   
USR-00005        1534.595455       4.181818       366.968478   

           spend_volatility  failed_txn_rate  retry_rate  dispute_rate  
user_id                                                                 
USR-00001          0.579754         0.053571    0.035714      0.000000  
USR-00002          0.587474         0.013514    0.013514      0.013514  
USR-00003          0.671421         0.050633    0.012658      0.037975  
USR-00004          0.720112         0.030303    0.015152      0.030303  
USR-00005          0.501935         0.021505    0.021505      0.021

## Spending Category Distribution (Lifestyle DNA)

In [8]:
# Spending Category Distribution
# Percentage of total spend per MCC category for purchases
category_spend = purchases.groupby(['user_id', 'categoria_mcc'])['monto'].sum().unstack(fill_value=0)

# Compute percentages
total_spend_per_user = category_spend.sum(axis=1)
lifestyle_dna = category_spend.div(total_spend_per_user, axis=0).fillna(0)

# Rename columns to pct_ prefixes
lifestyle_dna.columns = ['pct_' + col for col in lifestyle_dna.columns]

# Ensure all required categories are present (fill missing with 0)
required_cats = ['pct_supermercado', 'pct_restaurante', 'pct_entretenimiento', 'pct_viajes', 'pct_educacion', 'pct_salud', 'pct_tecnologia', 'pct_servicios_digitales', 'pct_ropa_accesorios', 'pct_transporte', 'pct_hogar', 'pct_gobierno', 'pct_transferencia', 'pct_delivery']
for cat in required_cats:
    if cat not in lifestyle_dna.columns:
        lifestyle_dna[cat] = 0

# Select only the specified features
selected_lifestyle = lifestyle_dna[['pct_supermercado', 'pct_restaurante', 'pct_entretenimiento', 'pct_viajes', 'pct_educacion', 'pct_salud', 'pct_tecnologia', 'pct_servicios_digitales', 'pct_ropa_accesorios', 'pct_transporte', 'pct_hogar']]

print("Lifestyle DNA features shape:", selected_lifestyle.shape)
print(selected_lifestyle.head())

Lifestyle DNA features shape: (14952, 11)
           pct_supermercado  pct_restaurante  pct_entretenimiento  pct_viajes  \
user_id                                                                         
USR-00001          0.062804         0.451993             0.227351    0.000000   
USR-00002          0.086723         0.498903             0.145168    0.000000   
USR-00003          0.030776         0.259852             0.159400    0.000000   
USR-00004          0.254392         0.155926             0.104638    0.165448   
USR-00005          0.071974         0.373844             0.349478    0.000000   

           pct_educacion  pct_salud  pct_tecnologia  pct_servicios_digitales  \
user_id                                                                        
USR-00001            0.0   0.000000        0.000000                 0.208067   
USR-00002            0.0   0.000000        0.025335                 0.106728   
USR-00003            0.0   0.000000        0.089951                 0.

## Credit Behavior Features

In [9]:
# Credit Behavior Features
# msi_usage_rate: proportion of purchases with MSI
msi_usage_rate = purchases.groupby('user_id')['meses_diferidos'].apply(lambda x: (x > 0).mean()).rename('msi_usage_rate')

# avg_msi_months: mean MSI months where used
avg_msi_months = purchases[purchases['meses_diferidos'] > 0].groupby('user_id')['meses_diferidos'].mean().rename('avg_msi_months')

# cashback_total: sum of cashback generated
cashback_total = df.groupby('user_id')['cashback_generado'].sum().rename('cashback_total')

# recurring_charge_count: count of recurring charges
recurring_charge_count = df[df['tipo_operacion'] == 'cargo_recurrente'].groupby('user_id').size().rename('recurring_charge_count')

# Combine credit behavior features
credit_behavior_features = pd.concat([msi_usage_rate, avg_msi_months, cashback_total, recurring_charge_count], axis=1).fillna(0)
print("Credit Behavior features shape:", credit_behavior_features.shape)
print(credit_behavior_features.head())

Credit Behavior features shape: (15025, 4)
           msi_usage_rate  avg_msi_months  cashback_total  \
user_id                                                     
USR-00001        0.000000             0.0          122.33   
USR-00002        0.000000             0.0          147.08   
USR-00003        0.000000             0.0          165.43   
USR-00004        0.148148             9.0            0.00   
USR-00005        0.000000             0.0          162.99   

           recurring_charge_count  
user_id                            
USR-00001                     9.0  
USR-00002                    12.0  
USR-00003                     8.0  
USR-00004                     5.0  
USR-00005                    17.0  


## Temporal Patterns Features

In [10]:
# Temporal Patterns Features
# night_owl_score: proportion of transactions between 22-5
night_hours = df['hora_del_dia'].isin(list(range(22, 24)) + list(range(0, 6)))
night_owl_score = df.groupby('user_id')['hora_del_dia'].apply(lambda x: night_hours.loc[x.index].mean()).rename('night_owl_score')

# weekend_spend_ratio: weekend spend / total spend
weekend_mask = df['dia_semana'].isin(['Saturday', 'Sunday'])
weekend_spend = df[weekend_mask].groupby('user_id')['monto'].sum()
total_spend = df.groupby('user_id')['monto'].sum()
weekend_spend_ratio = (weekend_spend / total_spend).fillna(0).rename('weekend_spend_ratio')

# peak_hour: mode of hora_del_dia
peak_hour = df.groupby('user_id')['hora_del_dia'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 0).rename('peak_hour')

# recency_days: days since last completed transaction
completed_txns = df[df['estatus'] == 'completada']
recency_days = (ref_date - completed_txns.groupby('user_id')['fecha_hora'].max()).dt.days.rename('recency_days')

# Combine temporal features
temporal_features = pd.concat([night_owl_score, weekend_spend_ratio, peak_hour, recency_days], axis=1).fillna(0)
print("Temporal Patterns features shape:", temporal_features.shape)
print(temporal_features.head())

Temporal Patterns features shape: (15025, 4)
           night_owl_score  weekend_spend_ratio  peak_hour  recency_days
user_id                                                                 
USR-00001         0.303571             0.299770          1            23
USR-00002         0.270270             0.198637         19            18
USR-00003         0.265823             0.083239         17            20
USR-00004         0.287879             0.469787          7            19
USR-00005         0.387097             0.366191          6            29


## Geography & Internationality Features

In [11]:
# Geography & Internationality Features
# international_txn_rate: proportion of international transactions
international_txn_rate = df.groupby('user_id')['es_internacional'].mean().rename('international_txn_rate')

# city_diversity: number of unique cities
city_diversity = df.groupby('user_id')['ciudad_transaccion'].nunique().rename('city_diversity')

# Note: out_of_state_rate omitted as home state data not available

# Combine geography features
geography_features = pd.concat([international_txn_rate, city_diversity], axis=1).fillna(0)
print("Geography & Internationality features shape:", geography_features.shape)
print(geography_features.head())

Geography & Internationality features shape: (15025, 2)
           international_txn_rate  city_diversity
user_id                                          
USR-00001                0.035714               5
USR-00002                0.013514               6
USR-00003                0.050633               9
USR-00004                0.030303              12
USR-00005                0.010753               8


## Cash vs Digital Features

In [12]:
# Cash vs Digital Features
# cash_dependency: proportion of cash-related transactions
cash_types = ['retiro_cajero', 'deposito_oxxo', 'deposito_farmacia']
cash_dependency = df.groupby('user_id')['tipo_operacion'].apply(lambda x: x.isin(cash_types).mean()).rename('cash_dependency')

# digital_payment_rate: proportion via digital channels
digital_channels = ['app_ios', 'app_android', 'app_huawei', 'codi']
digital_payment_rate = df.groupby('user_id')['canal'].apply(lambda x: x.isin(digital_channels).mean()).rename('digital_payment_rate')

# Combine cash vs digital features
cash_digital_features = pd.concat([cash_dependency, digital_payment_rate], axis=1).fillna(0)
print("Cash vs Digital features shape:", cash_digital_features.shape)
print(cash_digital_features.head())

Cash vs Digital features shape: (15025, 2)
           cash_dependency  digital_payment_rate
user_id                                         
USR-00001         0.035714              0.839286
USR-00002         0.027027              0.716216
USR-00003         0.050633              0.670886
USR-00004         0.015152              0.848485
USR-00005         0.064516              0.709677


## Risk/Anomaly Signals Features

In [13]:
# Risk/Anomaly Signals Features
# atypical_txn_rate: proportion of atypical transactions
atypical_txn_rate = df.groupby('user_id')['patron_uso_atipico'].mean().rename('atypical_txn_rate')

# large_txn_count: count of transactions above user's 95th percentile
def count_large_txns(group):
    if len(group) == 0:
        return 0
    pct95 = group['monto'].quantile(0.95)
    return (group['monto'] > pct95).sum()

large_txn_count = df.groupby('user_id').apply(count_large_txns).rename('large_txn_count')

# Combine risk features
risk_features = pd.concat([atypical_txn_rate, large_txn_count], axis=1).fillna(0)
print("Risk/Anomaly Signals features shape:", risk_features.shape)
print(risk_features.head())

Risk/Anomaly Signals features shape: (15025, 2)
           atypical_txn_rate  large_txn_count
user_id                                      
USR-00001                0.0                3
USR-00002                0.0                4
USR-00003                0.0                4
USR-00004                0.0                4
USR-00005                0.0                5


/tmp/ipykernel_102585/3158897664.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  large_txn_count = df.groupby('user_id').apply(count_large_txns).rename('large_txn_count')


## Final Feature Merge and Save

In [ ]:
# Merge all features into one dataframe
all_features = pd.concat([
    volume_freq_features,
    selected_lifestyle,
    credit_behavior_features,
    temporal_features,
    geography_features,
    cash_digital_features,
    risk_features
], axis=1, join='outer').fillna(0)

# Reset index to have user_id as column
all_features = all_features.reset_index()

print("Final features shape:", all_features.shape)
print("Columns:", list(all_features.columns))
print(all_features.head())

# Save to CSV
all_features.to_csv('user_features_transactions.csv', index=False)
print("Features saved to user_features.csv")

Final features shape: (15025, 33)
Columns: ['user_id', 'monthly_avg_spend', 'txn_frequency', 'avg_ticket_size', 'spend_volatility', 'failed_txn_rate', 'retry_rate', 'dispute_rate', 'pct_supermercado', 'pct_restaurante', 'pct_entretenimiento', 'pct_viajes', 'pct_educacion', 'pct_salud', 'pct_tecnologia', 'pct_servicios_digitales', 'pct_ropa_accesorios', 'pct_transporte', 'pct_hogar', 'msi_usage_rate', 'avg_msi_months', 'cashback_total', 'recurring_charge_count', 'night_owl_score', 'weekend_spend_ratio', 'peak_hour', 'recency_days', 'international_txn_rate', 'city_diversity', 'cash_dependency', 'digital_payment_rate', 'atypical_txn_rate', 'large_txn_count']
     user_id  monthly_avg_spend  txn_frequency  avg_ticket_size  \
0  USR-00001        1215.000000       2.818182       431.129032   
1  USR-00002        1468.359091       3.909091       375.626744   
2  USR-00003        1638.176364       3.909091       419.068372   
3  USR-00004        5661.518182       2.454545      2306.544444   
4

# Dataset result

In [16]:
df_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15025 entries, 0 to 15024
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   user_id                  15025 non-null  object 
 1   monthly_avg_spend        15025 non-null  float64
 2   txn_frequency            15025 non-null  float64
 3   avg_ticket_size          15025 non-null  float64
 4   spend_volatility         15025 non-null  float64
 5   failed_txn_rate          15025 non-null  float64
 6   retry_rate               15025 non-null  float64
 7   dispute_rate             15025 non-null  float64
 8   pct_supermercado         15025 non-null  float64
 9   pct_restaurante          15025 non-null  float64
 10  pct_entretenimiento      15025 non-null  float64
 11  pct_viajes               15025 non-null  float64
 12  pct_educacion            15025 non-null  float64
 13  pct_salud                15025 non-null  float64
 14  pct_tecnologia        